# Text-to-SQL Query Generation using an Encoder-Decoder Model (T5)

**Assignment:** Implement an encoder-decoder model to solve a real-world problem.

| | |
|---|---|
| **Problem** | Convert a natural-language question into an executable SQL query |
| **Model** | T5-small, a Transformer encoder-decoder (~60 M parameters), fine-tuned |
| **Dataset** | `b-mc2/sql-create-context` (~78k question / schema / SQL triples from WikiSQL + Spider) |
| **Metrics** | Exact Match, BLEU, Valid-SQL rate (the query really runs in SQLite) |
| **Runtime** | Google Colab, T4 GPU (`Runtime → Change runtime type → T4 GPU`) |

**Notebook outline**
1. Problem statement
2. Dataset
3. Model architecture
4. Setup
5. Data loading and exploration
6. Tokenization and preprocessing
7. Model loading and baseline (before training)
8. Training
9. Inference on custom questions
10. Evaluation
11. Error analysis and accuracy by query type
12. Visualising cross-attention
13. Interactive demo (optional)
14. Conclusion, limitations and future work

## 1. Problem Statement

Most business data is stored in relational databases, and getting answers out of them requires **SQL**. Managers, analysts, HR staff and other non-technical users usually can't write SQL, so even simple questions like *"How many employees are older than 30?"* have to go through a developer.

**Goal:** build a model that reads
- a **natural-language question**, and
- the **database schema** (`CREATE TABLE ...` statements),

and generates the **SQL query** that answers the question.

```
Input : "How many heads of the departments are older than 56?"
        schema: CREATE TABLE head (age INTEGER)
Output: SELECT COUNT(*) FROM head WHERE age > 56
```

This is a **sequence-to-sequence** problem: one variable-length sequence of text goes in, and another comes out in a different "language" (SQL). An **encoder-decoder** architecture is the natural fit. The encoder understands the question, and the decoder writes the query.

**Real-world applications:** natural-language interfaces to databases, "ask your data" features in BI dashboards (Power BI Q&A, Tableau Ask Data), chatbots over customer or HR records, and making data accessible to non-programmers.

## 2. Dataset: `b-mc2/sql-create-context`

- About **78,577** examples, built from two well-known Text-to-SQL benchmarks: **WikiSQL** (single-table questions over Wikipedia tables) and **Spider** (complex, multi-table queries).
- Every example has three fields:

| Field | Example |
|---|---|
| `question` | *How many heads of the departments are older than 56?* |
| `context` | `CREATE TABLE head (age INTEGER)` |
| `answer` | `SELECT COUNT(*) FROM head WHERE age > 56` |

**Why this dataset?** It provides the **schema together with every question**. The model learns to *use* the table and column names it is given rather than memorising them, so it works on tables it has never seen during training. It is licensed CC-BY-4.0 and loads directly from the Hugging Face Hub.

**Splits.** The dataset ships as one `train` split, so we carve out our own:

| Split | Size | Purpose |
|---|---|---|
| Train | 20,000 (subset, configurable) | Fine-tuning |
| Validation | 2,000 | Monitor loss, choose the best checkpoint |
| Test | 1,000 | Final evaluation, never seen during training |

## 3. Model Architecture: T5 (Text-to-Text Transfer Transformer)

T5 (Raffel et al., 2020) is a **Transformer encoder-decoder** that frames every NLP task as *text in → text out*. Text-to-SQL fits this exactly.

```
 "translate to SQL: question: ... schema: CREATE TABLE ..."
                         │
                ┌────────▼─────────┐
                │ Token Embedding   │   SentencePiece sub-word tokens
                └────────┬─────────┘
                ┌────────▼─────────┐
                │     ENCODER       │   6 × [ Self-Attention → Feed-Forward ]
                │  (bidirectional)  │   reads the whole question + schema at once
                └────────┬─────────┘
                         │  one context vector per input token
                         ▼
                ┌──────────────────┐
  SQL tokens ─► │     DECODER       │   6 × [ Masked Self-Attention
  so far        │ (autoregressive)  │         → Cross-Attention (to encoder)
                └────────┬─────────┘         → Feed-Forward ]
                ┌────────▼─────────┐
                │ Linear + Softmax  │   probability of every vocabulary token
                └────────┬─────────┘
                         ▼
          "SELECT COUNT(*) FROM head WHERE age > 56"   (one token at a time)
```

| Hyper-parameter | `t5-small` |
|---|---|
| Parameters | ~60 M |
| Encoder layers / Decoder layers | 6 / 6 |
| Hidden size (d_model) | 512 |
| Attention heads | 8 |
| Feed-forward size | 2048 |
| Vocabulary | 32,100 SentencePiece tokens |

**Role of each part**
- **Encoder:** turns the question and schema into contextual vectors. Self-attention lets words like *"older than"* connect to the column `age`, and *"how many"* connect to `COUNT`.
- **Decoder:** generates the SQL **auto-regressively**. *Masked self-attention* looks at the SQL written so far, and *cross-attention* looks back at the encoder output to pick which table, column or value to copy next. (Section 12 visualises this.)
- **Training:** uses **teacher forcing**. At each step the decoder receives the *correct* previous SQL token, and cross-entropy loss is computed on its prediction of the next one.
- **Inference:** uses **beam search** with 4 beams, which keeps the 4 most probable partial queries at every step and returns the best complete one.

**Why fine-tune a pretrained model?** T5 was pretrained on the large C4 English corpus, so it already "understands" English. Fine-tuning only has to teach it the mapping from English to SQL. This **transfer learning** gives far higher accuracy than training an encoder-decoder from scratch with the same data and compute.

## 4. Setup

Install the libraries. Colab already has PyTorch; `transformers` provides T5, `datasets` loads the data, and `nltk` computes BLEU.

In [ ]:
!pip install -q transformers datasets accelerate sentencepiece nltk

Imports and all hyper-parameters in one place. To trade accuracy for speed, change `TRAIN_SAMPLES` and `EPOCHS`.

In [ ]:
import re, time, random, sqlite3, warnings
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments, Seq2SeqTrainer, set_seed,
)
warnings.filterwarnings("ignore")

# ---------------- configuration ----------------
MODEL_NAME     = "google-t5/t5-small"
TRAIN_SAMPLES  = 20000      # raise to 50000+ for better accuracy if time allows
VAL_SAMPLES    = 2000
TEST_SAMPLES   = 1000
MAX_INPUT_LEN  = 256        # tokens for question + schema
MAX_TARGET_LEN = 128        # tokens for the SQL query
EPOCHS         = 3
BATCH_SIZE     = 16
LR             = 3e-4       # standard learning rate for fine-tuning T5
NUM_BEAMS      = 4
SEED           = 42

set_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", torch.cuda.get_device_name(0) if device == "cuda" else "CPU (training will be very slow!)")

# ---------------- plot style ----------------
BLUE, ORANGE, INK, MUTED = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#b5b4ad", "axes.labelcolor": MUTED,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.grid": True, "grid.color": "#e4e3de", "grid.linewidth": 0.8,
    "axes.titleweight": "bold", "axes.titlecolor": INK, "font.size": 10,
    "lines.linewidth": 2, "legend.frameon": False,
})

## 5. Data Loading and Exploration

Download the dataset and create **train / validation / test** splits with a fixed seed so results are reproducible.

In [ ]:
raw = load_dataset("b-mc2/sql-create-context")
print(raw)

split    = raw["train"].train_test_split(test_size=0.1, seed=SEED)
val_test = split["test"].train_test_split(test_size=0.5, seed=SEED)

train_ds = split["train"].shuffle(seed=SEED).select(range(TRAIN_SAMPLES))
val_ds   = val_test["train"].select(range(VAL_SAMPLES))
test_ds  = val_test["test"].select(range(TEST_SAMPLES))

print(f"\nTrain: {len(train_ds):,} | Validation: {len(val_ds):,} | Test: {len(test_ds):,}")

A few raw examples:

In [ ]:
for ex in train_ds.select(range(4)):
    print("QUESTION:", ex["question"])
    print("SCHEMA  :", ex["context"])
    print("SQL     :", ex["answer"])
    print("-" * 90)

**Length statistics.** These guide the choice of `MAX_INPUT_LEN` and `MAX_TARGET_LEN`. Almost all inputs are short, so 256 input tokens and 128 target tokens cover nearly everything without wasting GPU memory.

In [ ]:
q_len = [len(x.split()) for x in train_ds["question"]]
c_len = [len(x.split()) for x in train_ds["context"]]
a_len = [len(x.split()) for x in train_ds["answer"]]

fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
for ax, data, title in zip(axes, [q_len, c_len, a_len], ["Question", "Schema", "SQL query"]):
    ax.hist(data, bins=40, color=BLUE, edgecolor="#fcfcfb", linewidth=1)
    ax.axvline(np.percentile(data, 95), color=INK, linestyle="--", linewidth=1)
    ax.text(np.percentile(data, 95), ax.get_ylim()[1] * 0.92, "  95th pct", color=MUTED, fontsize=9)
    ax.set_title(f"{title} length"); ax.set_xlabel("words"); ax.set_ylabel("examples")
plt.tight_layout(); plt.show()

for name, data in [("Question", q_len), ("Schema", c_len), ("SQL", a_len)]:
    print(f"{name:9s} mean = {np.mean(data):5.1f} words | 95th pct = {np.percentile(data, 95):4.0f} | max = {max(data)}")

**What kinds of SQL queries are in the data?** We sort each query into a type. The same function is reused in Section 11 to measure accuracy per type.

In [ ]:
def query_type(sql):
    s = " " + sql.upper() + " "
    if " JOIN " in s:                                   return "JOIN"
    if "GROUP BY" in s or "ORDER BY" in s:              return "GROUP / ORDER BY"
    if re.search(r"\b(COUNT|SUM|AVG|MIN|MAX)\s*\(", s): return "Aggregate"
    if " WHERE " in s:
        return "WHERE (multi-cond)" if re.search(r"\b(AND|OR)\b", s) else "WHERE (1 cond)"
    return "Simple SELECT"

type_counts = pd.Series([query_type(a) for a in train_ds["answer"]]).value_counts()

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.barh(type_counts.index[::-1], type_counts.values[::-1], color=BLUE, height=0.6)
for i, v in enumerate(type_counts.values[::-1]):
    ax.text(v, i, f"  {v:,} ({v / len(train_ds):.0%})", va="center", color=INK, fontsize=9)
ax.set_title("Query types in the training set"); ax.set_xlabel("examples")
ax.grid(axis="y", visible=False); ax.set_xlim(0, type_counts.max() * 1.25)
plt.tight_layout(); plt.show()

## 6. Tokenization and Preprocessing

### 6.1 Tokenizer, and fixing the missing `<` symbol
T5 uses a SentencePiece tokenizer that splits text into sub-word pieces. Its vocabulary has **no `<` character**: it becomes `<unk>` (unknown), so the model could never write `WHERE age < 30`. We detect this and **add `<` as a new token**. Section 7 resizes the model's embedding matrix to match.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

sample = "SELECT name FROM people WHERE age < 30"
print("Before fix:", tokenizer.convert_ids_to_tokens(tokenizer(sample).input_ids))

if tokenizer.unk_token_id in tokenizer("<").input_ids:
    tokenizer.add_tokens(["<"])
    print("Added '<' to the vocabulary.")

print("After fix :", tokenizer.convert_ids_to_tokens(tokenizer(sample).input_ids))
print("Vocabulary size:", len(tokenizer))

### 6.2 Input format
Question and schema are joined into **one input string** with a task prefix (the T5 convention):

```
translate to SQL: question: <question> schema: <CREATE TABLE ...>
```

The **target** (label) is the SQL string. Both are tokenized and truncated to their maximum lengths. Padding happens later, per batch, to save memory.

In [ ]:
def build_input(question, context):
    return f"translate to SQL: question: {question} schema: {context}"

def preprocess(batch):
    inputs = [build_input(q, c) for q, c in zip(batch["question"], batch["context"])]
    model_inputs = tokenizer(inputs, max_length=MAX_INPUT_LEN, truncation=True)
    labels = tokenizer(text_target=batch["answer"], max_length=MAX_TARGET_LEN, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tok_train = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
tok_val   = val_ds.map(preprocess,   batched=True, remove_columns=val_ds.column_names)
print(tok_train)

ex = tok_train[0]
print("\nEncoder input :", tokenizer.decode(ex["input_ids"]))
print("Decoder target:", tokenizer.decode(ex["labels"]))

## 7. Model Loading and Baseline

Load pretrained **T5-small**. Its embedding matrix is resized because we added the `<` token.

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model.resize_token_embeddings(len(tokenizer))
model.to(device)

n_params = sum(p.numel() for p in model.parameters())
cfg = model.config
print(f"Parameters      : {n_params / 1e6:.1f} M")
print(f"Encoder layers  : {cfg.num_layers}")
print(f"Decoder layers  : {cfg.num_decoder_layers}")
print(f"Hidden size     : {cfg.d_model}")
print(f"Attention heads : {cfg.num_heads}")

### 7.1 Inference helper
`generate_sql` encodes the question and schema with the **encoder**, then lets the **decoder** generate SQL tokens with beam search until it emits the end-of-sequence token. `clean_sql` tidies the spacing around operators, since the newly added `<` token can lose its surrounding spaces when decoded.

In [ ]:
def clean_sql(s):
    s = re.sub(r"\s*(<=|>=|!=|<>|=|<|>)\s*", r" \1 ", s)
    return " ".join(s.split())

def generate_sql(questions, contexts, num_beams=NUM_BEAMS, batch_size=32):
    model.eval()
    preds = []
    for i in range(0, len(questions), batch_size):
        inputs = [build_input(q, c) for q, c in
                  zip(questions[i:i + batch_size], contexts[i:i + batch_size])]
        enc = tokenizer(inputs, max_length=MAX_INPUT_LEN, truncation=True,
                        padding=True, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=MAX_TARGET_LEN, num_beams=num_beams)
        preds.extend(clean_sql(p) for p in tokenizer.batch_decode(out, skip_special_tokens=True))
    return preds

### 7.2 Baseline: the model *before* fine-tuning
Pretrained T5 has never been taught SQL, so its output should be meaningless. This shows that everything the model learns about SQL comes from our training.

In [ ]:
demo = test_ds.select(range(3))
for q, c, g, p in zip(demo["question"], demo["context"], demo["answer"],
                      generate_sql(demo["question"], demo["context"])):
    print(f"QUESTION : {q}\nGOLD SQL : {g}\nPREDICTED: {p}\n")

## 8. Training (Fine-tuning)

- **Loss:** token-level cross-entropy with teacher forcing.
- **Optimizer:** AdamW, LR = 3e-4, linear decay with 200 warm-up steps, weight decay 0.01.
- **`DataCollatorForSeq2Seq`:** pads each batch to its longest sequence and sets padded label positions to `-100` so they are ignored by the loss.
- **Model selection:** validation loss is computed after every epoch, and the best checkpoint is restored at the end.
- **`fp16` is off:** T5 is known to give `NaN` losses in half precision.

In [ ]:
collator = DataCollatorForSeq2Seq(tokenizer, model=model)

args = Seq2SeqTrainingArguments(
    output_dir="t5-text2sql",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    warmup_steps=200,
    logging_steps=100,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=False,
    report_to="none",
    seed=SEED,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tok_train,
    eval_dataset=tok_val,
    data_collator=collator,
    processing_class=tokenizer,
)

Start training. On a T4 GPU with the default settings this takes roughly 15–30 minutes.

In [ ]:
train_result = trainer.train()
train_minutes = train_result.metrics["train_runtime"] / 60
print(f"\nTraining finished in {train_minutes:.1f} minutes")

### 8.1 Training curve
Train loss (logged every 100 steps) and validation loss (after each epoch). Both falling together, with no gap opening up, means the model is learning without over-fitting.

In [ ]:
hist = trainer.state.log_history
tr = [(h["step"], h["loss"])      for h in hist if "loss" in h]
ev = [(h["step"], h["eval_loss"]) for h in hist if "eval_loss" in h]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(*zip(*tr), color=BLUE, label="Train loss")
ax.plot(*zip(*ev), color=ORANGE, marker="o", markersize=8,
        markeredgecolor="#fcfcfb", markeredgewidth=2, label="Validation loss")
ax.annotate(f"{ev[-1][1]:.3f}", ev[-1], textcoords="offset points", xytext=(6, 8), color=INK, fontsize=9)
ax.set_title("Training curve: T5-small on Text-to-SQL")
ax.set_xlabel("training step"); ax.set_ylabel("cross-entropy loss")
ax.set_ylim(bottom=0); ax.legend()
plt.tight_layout(); plt.show()

final_train_loss = tr[-1][1]
best_val_loss = min(v for _, v in ev)
print(f"Final train loss: {final_train_loss:.4f} | Best validation loss: {best_val_loss:.4f}")

Save the fine-tuned model and tokenizer so they can be reloaded later with `from_pretrained("t5-text2sql-final")`.

In [ ]:
trainer.save_model("t5-text2sql-final")
tokenizer.save_pretrained("t5-text2sql-final")
print("Saved to ./t5-text2sql-final")

## 9. Inference on Custom Questions

The real test of generalisation: a **table the model has never seen**, with questions written by us.

In [ ]:
schema = ("CREATE TABLE employees (name VARCHAR, department VARCHAR, "
          "salary INTEGER, age INTEGER, city VARCHAR)")
questions = [
    "How many employees are older than 30?",
    "What is the average salary of employees in the Sales department?",
    "List the names of employees who live in Delhi.",
    "What is the highest salary of employees younger than 25?",
    "Which department does Rahul work in?",
    "Show the names and salaries of employees in Mumbai ordered by salary.",
]
print("SCHEMA:", schema, "\n")
for q, sql in zip(questions, generate_sql(questions, [schema] * len(questions))):
    print(f"Q  : {q}\nSQL: {sql}\n")

**Running the generated SQL on a real database.** We create the `employees` table in SQLite, insert a few rows and execute the model's queries. This shows the output is real, executable SQL.

In [ ]:
conn = sqlite3.connect(":memory:")
conn.execute(schema)
conn.executemany("INSERT INTO employees VALUES (?, ?, ?, ?, ?)", [
    ("Rahul",  "Sales",     52000, 28, "Delhi"),
    ("Priya",  "HR",        61000, 34, "Mumbai"),
    ("Amit",   "Sales",     47000, 23, "Delhi"),
    ("Sneha",  "IT",        88000, 41, "Bangalore"),
    ("Karan",  "IT",        39000, 22, "Mumbai"),
])
for q in questions[:4]:
    sql = generate_sql([q], [schema])[0]
    try:
        result = conn.execute(sql).fetchall()
    except sqlite3.Error as e:
        result = f"ERROR: {e}"
    print(f"Q     : {q}\nSQL   : {sql}\nRESULT: {result}\n")
conn.close()

## 10. Evaluation on the Held-out Test Set

| Metric | What it measures |
|---|---|
| **Exact Match (EM)** | Predicted SQL equals the gold SQL after normalisation (case, spacing, trailing `;`). The strictest metric. |
| **BLEU** | n-gram overlap between predicted and gold SQL. Gives partial credit for nearly correct queries. |
| **Valid-SQL rate** | The predicted query **executes** in SQLite against the given schema, with no syntax errors or unknown tables or columns. |

In [ ]:
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

def normalize_sql(s):
    s = s.strip().rstrip(";").lower()
    s = re.sub(r"\s*(<=|>=|!=|<>|=|<|>|,|\(|\))\s*", r" \1 ", s)
    return " ".join(s.split())

def is_valid_sql(query, context):
    """Build the schema in an in-memory SQLite DB and check that the query executes."""
    conn = sqlite3.connect(":memory:")
    try:
        for stmt in re.split(r"(?i)(?=CREATE TABLE)", context):
            stmt = stmt.strip().rstrip(",;").strip()
            if stmt:
                conn.execute(stmt)
        conn.execute(query)
        return True
    except Exception:
        return False
    finally:
        conn.close()

In [ ]:
questions_t, contexts_t, golds_t = test_ds["question"], test_ds["context"], test_ds["answer"]

t0 = time.time()
preds_t = generate_sql(questions_t, contexts_t)
print(f"Generated {len(preds_t)} queries in {time.time() - t0:.0f} s")

exact   = [normalize_sql(p) == normalize_sql(g) for p, g in zip(preds_t, golds_t)]
gold_ok = [is_valid_sql(g, c) for g, c in zip(golds_t, contexts_t)]
pred_ok = [is_valid_sql(p, c) for p, c in zip(preds_t, contexts_t)]

em    = np.mean(exact)
bleu  = corpus_bleu([[normalize_sql(g).split()] for g in golds_t],
                    [normalize_sql(p).split() for p in preds_t],
                    smoothing_function=SmoothingFunction().method1)
# Count only samples whose gold query runs (a few dataset schemas are imperfect)
valid = np.mean([p for p, g in zip(pred_ok, gold_ok) if g])

print(f"\nExact Match    : {em * 100:.2f} %")
print(f"BLEU           : {bleu * 100:.2f}")
print(f"Valid-SQL rate : {valid * 100:.2f} %   (on {sum(gold_ok)} samples whose gold SQL runs)")

**Results summary.** This prints a ready-to-copy markdown table for the report. All predictions are also saved to `test_predictions.csv`.

In [ ]:
results_df = pd.DataFrame({
    "question": questions_t, "schema": contexts_t, "gold_sql": golds_t, "pred_sql": preds_t,
    "query_type": [query_type(g) for g in golds_t],
    "exact_match": exact, "pred_valid_sql": pred_ok,
})
results_df.to_csv("test_predictions.csv", index=False)

summary = {
    "Model": f"t5-small ({n_params / 1e6:.0f} M params, {cfg.num_layers}+{cfg.num_decoder_layers} layers)",
    "Training samples": f"{TRAIN_SAMPLES:,}",
    "Epochs": EPOCHS,
    "Final train loss": f"{final_train_loss:.4f}",
    "Best validation loss": f"{best_val_loss:.4f}",
    "Test Exact Match": f"{em * 100:.2f} %",
    "Test BLEU": f"{bleu * 100:.2f}",
    "Test Valid-SQL rate": f"{valid * 100:.2f} %",
    "Training time (GPU)": f"{train_minutes:.1f} min",
}
print("| Setting | Value |\n|---|---|")
for k, v in summary.items():
    print(f"| {k} | {v} |")

## 11. Error Analysis

### 11.1 Accuracy by query type
Simple single-table queries should be easy. Multi-condition and multi-table (JOIN) queries are expected to be harder.

In [ ]:
by_type = (results_df.groupby("query_type")
           .agg(n=("exact_match", "size"), exact_match=("exact_match", "mean"),
                valid_sql=("pred_valid_sql", "mean"))
           .sort_values("exact_match"))
display(by_type.style.format({"exact_match": "{:.1%}", "valid_sql": "{:.1%}"}))

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.barh(by_type.index, by_type["exact_match"] * 100, color=BLUE, height=0.6)
for i, (v, n) in enumerate(zip(by_type["exact_match"] * 100, by_type["n"])):
    ax.text(v, i, f"  {v:.1f}%  (n={n})", va="center", color=INK, fontsize=9)
ax.set_xlim(0, 115); ax.set_xlabel("exact match (%)")
ax.set_title("Test exact match by query type"); ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

### 11.2 Correct and incorrect predictions

In [ ]:
print("=" * 30, "CORRECT PREDICTIONS", "=" * 30)
for _, r in results_df[results_df.exact_match].head(5).iterrows():
    print(f"Q   : {r.question}\nGOLD: {r.gold_sql}\nPRED: {r.pred_sql}\n")

print("=" * 30, "WRONG PREDICTIONS", "=" * 30)
for _, r in results_df[~results_df.exact_match].head(6).iterrows():
    print(f"Q     : {r.question}\nSCHEMA: {r.schema}\nGOLD  : {r.gold_sql}\nPRED  : {r.pred_sql}"
          f"\nVALID : {r.pred_valid_sql}\n")

**Typical error patterns** (check them against the output above):
1. **Value mismatch:** correct structure, but a literal differs slightly (for example, capitalisation or a partly copied string). These are often *nearly* correct, which is why BLEU stays high even when EM fails.
2. **Wrong aggregate or column:** `MAX` instead of `COUNT`, or a similarly named column, usually when the question is ambiguous.
3. **Multi-table queries:** missing or incorrect `JOIN ... ON` conditions. Spider-style questions are much harder, and they are a small share of the training data.
4. **Noisy gold labels:** some WikiSQL-derived references are themselves odd, so a "wrong" prediction is sometimes a valid alternative.

## 12. Visualising Cross-Attention: How the Decoder Uses the Encoder

Every time the decoder writes a SQL token, its **cross-attention** layer weighs all of the encoder's input tokens. The heatmap shows, for each generated SQL token (rows), how strongly the model attends to each input token (columns), averaged over the 8 heads of the last decoder layer. Look for bright cells where the model *copies* a column name from the schema, or links *"how many"* to `COUNT`.

In [ ]:
def plot_cross_attention(question, context, layer=-1):
    model.eval()
    enc = tokenizer(build_input(question, context), return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=MAX_TARGET_LEN, num_beams=1)
        res = model(**enc, decoder_input_ids=out[:, :-1], output_attentions=True)
    if not res.cross_attentions:
        print("This transformers version did not return attentions; reload the model with "
              "AutoModelForSeq2SeqLM.from_pretrained(..., attn_implementation='eager').")
        return
    att = res.cross_attentions[layer][0].mean(0).float().cpu().numpy()   # (sql_tokens, input_tokens)
    src = [t.replace("▁", "") or "_" for t in tokenizer.convert_ids_to_tokens(enc.input_ids[0])]
    tgt = [t.replace("▁", "") or "_" for t in tokenizer.convert_ids_to_tokens(out[0, 1:])]

    fig, ax = plt.subplots(figsize=(max(8, len(src) * 0.32), max(3.5, len(tgt) * 0.32)))
    im = ax.imshow(att, cmap="Blues", aspect="auto")
    ax.set_xticks(range(len(src))); ax.set_xticklabels(src, rotation=90, fontsize=8)
    ax.set_yticks(range(len(tgt))); ax.set_yticklabels(tgt, fontsize=8)
    ax.grid(False)
    ax.set_xlabel("encoder input tokens (question + schema)"); ax.set_ylabel("generated SQL tokens")
    ax.set_title("Decoder cross-attention (last layer, mean of heads)")
    fig.colorbar(im, ax=ax, fraction=0.025, pad=0.01, label="attention weight")
    plt.tight_layout(); plt.show()
    print("Generated SQL:", clean_sql(tokenizer.decode(out[0], skip_special_tokens=True)))

plot_cross_attention("How many employees are older than 30?",
                     "CREATE TABLE employees (name VARCHAR, salary INTEGER, age INTEGER)")

## 13. Interactive Demo (optional)

A small web UI built with Gradio. Run the cell, open the public link and try your own questions and schemas. It's useful for screenshots in the report.

In [ ]:
!pip install -q gradio
import gradio as gr

def demo_fn(question, schema):
    return generate_sql([question], [schema])[0]

gr.Interface(
    fn=demo_fn,
    inputs=[gr.Textbox(label="Question", value="How many employees earn more than 50000?"),
            gr.Textbox(label="Schema (CREATE TABLE ...)", lines=3,
                       value="CREATE TABLE employees (name VARCHAR, department VARCHAR, salary INTEGER, age INTEGER)")],
    outputs=gr.Textbox(label="Generated SQL"),
    title="Text-to-SQL with T5 (Encoder-Decoder)",
    description="Type a question in English and the table schema; the fine-tuned T5 model writes the SQL.",
).launch(share=True)

Optional: download the trained model and the prediction file to your computer.

In [ ]:
import shutil
shutil.make_archive("t5-text2sql-final", "zip", "t5-text2sql-final")
try:
    from google.colab import files
    files.download("t5-text2sql-final.zip")
    files.download("test_predictions.csv")
except ImportError:
    print("Not running in Colab; files are in the current directory.")

## 14. Conclusion

- We framed **Text-to-SQL** as a sequence-to-sequence problem and solved it with a **Transformer encoder-decoder (T5-small)**, fine-tuned on 20k examples of the `sql-create-context` dataset (WikiSQL + Spider).
- The **encoder** builds contextual representations of the question and schema. The **decoder** generates SQL token by token, using **cross-attention** to copy the right tables, columns and values from the input (Section 12).
- Passing the **schema as part of the input** lets the model generalise to tables it has never seen (Section 9), and the generated queries **run on a real SQLite database**.
- Before fine-tuning the model produced no usable SQL (Section 7.2). After about 20 minutes of fine-tuning it produces executable SQL for most test questions (see the metrics in Section 10).

### Limitations
- Weaker on complex queries: multi-table `JOIN`s, nested sub-queries, `GROUP BY ... HAVING`.
- Exact Match is strict. A semantically equivalent query written differently counts as wrong.
- Valid-SQL only checks that the query *runs*, not that it returns the *right* rows.
- Generated SQL should never run with write permissions in production, because the model could produce `DELETE`/`DROP` statements.

### Future work
- Larger models (`t5-base`, `flan-t5-base`, `CodeT5`) and the full 78k training set.
- **Execution accuracy** on real populated databases (the Spider DBs), comparing returned rows.
- **Constrained decoding** (for example, PICARD) so only grammatically valid SQL can be produced.
- Multi-turn conversational Text-to-SQL (SParC, CoSQL datasets).

### References
1. C. Raffel et al., *Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer*, JMLR 2020.
2. V. Zhong, C. Xiong, R. Socher, *Seq2SQL: Generating Structured Queries from Natural Language using Reinforcement Learning*, 2017 (WikiSQL).
3. T. Yu et al., *Spider: A Large-Scale Human-Labeled Dataset for Complex and Cross-Domain Semantic Parsing and Text-to-SQL Task*, EMNLP 2018.
4. A. Vaswani et al., *Attention Is All You Need*, NeurIPS 2017.
5. Dataset: `b-mc2/sql-create-context`, Hugging Face Hub (CC-BY-4.0).